# 03 — HTTPX Async

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- utiliser `httpx.AsyncClient` pour des requêtes HTTP asynchrones ;
- gérer les sessions, les timeouts et les retries ;
- paralléliser des appels API avec `gather` et un sémaphore ;
- streamer des réponses volumineuses ;
- comparer les performances sync vs async.

## Prérequis — ce que vous connaissez déjà

Vous maîtrisez déjà :

- `async def`, `await`, `asyncio.gather()`, `create_task()` ;
- l'event loop et le modèle coopératif ;
- `asyncio.Semaphore` pour limiter la concurrence ;
- les bases de HTTP (méthodes, status codes, headers).

Notions introduites ici :

- `httpx.AsyncClient` et ses méthodes (`get`, `post`, etc.) ;
- gestion des sessions, du pooling et des timeouts ;
- streaming de réponses.

## Plan

1. Pourquoi httpx ?
2. Première requête async
3. `AsyncClient` comme context manager
4. Requêtes parallèles avec gather
5. Rate limiting avec Semaphore
6. Timeouts et retries
7. Streaming de réponses
8. POST, headers, authentification
9. Benchmark sync vs async
10. Synthèse
11. Exercices

---

## 1. Pourquoi httpx ?

`httpx` est le remplaçant moderne de `requests` avec support **natif async**.

| Feature | `requests` | `httpx` |
|---|---|---|
| API synchrone | Oui | Oui |
| API asynchrone | Non | **Oui** |
| HTTP/2 | Non | Oui |
| Timeouts par défaut | Non | Oui |
| Type hints | Partiel | Complet |

```bash
pip install httpx
# ou
uv add httpx
```

In [ ]:
import httpx

print(f"httpx version : {httpx.__version__}")

---

## 2. Première requête async

In [ ]:
import httpx

# Requête simple (crée un client à usage unique)
async with httpx.AsyncClient() as client:
    response = await client.get("https://httpbin.org/get")

print(f"Status : {response.status_code}")
print(f"Content-Type : {response.headers['content-type']}")
print(f"Body (premiers 200 chars) : {response.text[:200]}")

### L'objet `Response`

| Attribut | Description |
|---|---|
| `.status_code` | Code HTTP (200, 404, etc.) |
| `.headers` | Headers de réponse |
| `.text` | Corps en texte |
| `.json()` | Corps parsé en JSON |
| `.content` | Corps en bytes |
| `.raise_for_status()` | Lève une exception si erreur |

In [ ]:
import httpx

async with httpx.AsyncClient() as client:
    r = await client.get("https://httpbin.org/json")
    data = r.json()
    print(f"Clés : {list(data.keys())}")

---

## 3. `AsyncClient` comme context manager

Utilisez toujours `async with` pour gérer le cycle de vie du client (pooling de connexions, fermeture propre).

In [ ]:
import httpx

# Bon : le pool de connexions est réutilisé
async with httpx.AsyncClient(base_url="https://httpbin.org") as client:
    r1 = await client.get("/get")
    r2 = await client.get("/headers")
    print(f"Requête 1 : {r1.status_code}")
    print(f"Requête 2 : {r2.status_code}")

### `base_url` pour simplifier les URLs

In [ ]:
import httpx

async with httpx.AsyncClient(
    base_url="https://httpbin.org",
    headers={"User-Agent": "MonApp/1.0"},
) as client:
    r = await client.get("/user-agent")
    print(r.json())

---

## 4. Requêtes parallèles avec gather

In [ ]:
import asyncio
import httpx
import time

async def fetch(client: httpx.AsyncClient, url: str) -> int:
    r = await client.get(url)
    return r.status_code

urls = [f"https://httpbin.org/delay/{i % 3}" for i in range(6)]

start = time.perf_counter()
async with httpx.AsyncClient() as client:
    statuses = await asyncio.gather(*(fetch(client, url) for url in urls))
elapsed = time.perf_counter() - start

print(f"Status codes : {statuses}")
print(f"Temps : {elapsed:.2f}s (bien moins que séquentiel)")

---

## 5. Rate limiting avec Semaphore

Pour respecter les limites d'une API, on combine `asyncio.Semaphore` avec les requêtes.

In [ ]:
import asyncio
import httpx
import time

sem = asyncio.Semaphore(3)  # max 3 requêtes simultanées

async def fetch_limited(client: httpx.AsyncClient, url: str) -> int:
    async with sem:
        r = await client.get(url)
        return r.status_code

urls = [f"https://httpbin.org/delay/1" for _ in range(9)]

start = time.perf_counter()
async with httpx.AsyncClient() as client:
    statuses = await asyncio.gather(*(fetch_limited(client, url) for url in urls))
elapsed = time.perf_counter() - start

print(f"9 requêtes avec sem(3) : {elapsed:.2f}s (≈3s au lieu de 9s)")

---

## 6. Timeouts et retries

In [ ]:
import httpx

# Timeout granulaire
timeout = httpx.Timeout(
    connect=5.0,   # connexion
    read=10.0,     # lecture
    write=5.0,     # écriture
    pool=5.0,      # attente d'une connexion du pool
)

async with httpx.AsyncClient(timeout=timeout) as client:
    try:
        r = await client.get("https://httpbin.org/delay/2")
        print(f"OK : {r.status_code}")
    except httpx.TimeoutException as e:
        print(f"Timeout : {e}")

### Retry pattern

In [ ]:
import asyncio
import httpx

async def fetch_with_retry(
    client: httpx.AsyncClient,
    url: str,
    max_retries: int = 3,
    backoff: float = 1.0,
) -> httpx.Response:
    for attempt in range(1, max_retries + 1):
        try:
            r = await client.get(url)
            r.raise_for_status()
            return r
        except (httpx.HTTPStatusError, httpx.TimeoutException) as e:
            if attempt == max_retries:
                raise
            wait = backoff * (2 ** (attempt - 1))  # exponential backoff
            print(f"  Retry {attempt}/{max_retries} après {wait}s : {e}")
            await asyncio.sleep(wait)

async with httpx.AsyncClient() as client:
    try:
        r = await fetch_with_retry(client, "https://httpbin.org/status/200")
        print(f"Succès : {r.status_code}")
    except httpx.HTTPStatusError as e:
        print(f"Échec final : {e}")

---

## 7. Streaming de réponses

Pour les réponses volumineuses, streamer évite de tout charger en mémoire.

In [ ]:
import httpx

async with httpx.AsyncClient() as client:
    async with client.stream("GET", "https://httpbin.org/stream/5") as response:
        async for line in response.aiter_lines():
            print(f"  Chunk : {line[:80]}...")

### Télécharger un fichier par chunks

In [ ]:
import httpx

async with httpx.AsyncClient() as client:
    async with client.stream("GET", "https://httpbin.org/bytes/1024") as response:
        total = 0
        async for chunk in response.aiter_bytes(chunk_size=256):
            total += len(chunk)
        print(f"Total téléchargé : {total} bytes")

---

## 8. POST, headers, authentification

In [ ]:
import httpx

async with httpx.AsyncClient() as client:
    # POST avec JSON
    r = await client.post(
        "https://httpbin.org/post",
        json={"nom": "Alice", "age": 30},
    )
    data = r.json()
    print(f"Envoyé : {data['json']}")

In [ ]:
import httpx

# Headers personnalisés et authentification
async with httpx.AsyncClient(
    headers={"X-Custom": "valeur"},
    auth=("user", "pass"),
) as client:
    r = await client.get("https://httpbin.org/basic-auth/user/pass")
    print(f"Auth : {r.json()}")

---

## 9. Benchmark sync vs async

In [ ]:
import asyncio
import httpx
import time

URL = "https://httpbin.org/delay/1"
N = 5

# Synchrone
start = time.perf_counter()
with httpx.Client() as client:
    for _ in range(N):
        client.get(URL)
t_sync = time.perf_counter() - start

# Asynchrone
start = time.perf_counter()
async with httpx.AsyncClient() as client:
    await asyncio.gather(*(client.get(URL) for _ in range(N)))
t_async = time.perf_counter() - start

print(f"Sync  : {t_sync:.2f}s")
print(f"Async : {t_async:.2f}s")
print(f"Speedup : {t_sync / t_async:.1f}x")

---

## 10. Synthèse

| Concept | Code |
|---|---|
| Client async | `async with httpx.AsyncClient() as client:` |
| GET | `await client.get(url)` |
| POST JSON | `await client.post(url, json=data)` |
| Parallèle | `gather(*(client.get(u) for u in urls))` |
| Rate limit | `asyncio.Semaphore(n)` |
| Timeout | `httpx.Timeout(connect=5, read=10)` |
| Streaming | `async with client.stream() as r:` |
| Retry | Boucle + exponential backoff |

**Bonnes pratiques :**

1. Toujours utiliser `async with` pour le client.
2. Limiter la concurrence avec un sémaphore.
3. Configurer les timeouts explicitement.
4. Utiliser le streaming pour les gros volumes.

---

## 11. Exercices

### Exercice 1 — Fetch multiple *(facile)*

Utiliser `httpx.AsyncClient` pour faire 5 requêtes GET vers `https://httpbin.org/uuid` en parallèle. Afficher chaque UUID reçu.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Httpx_async", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import httpx

async with httpx.AsyncClient() as client:
    responses = await asyncio.gather(
        *(client.get("https://httpbin.org/uuid") for _ in range(5))
    )

for i, r in enumerate(responses):
    print(f"  {i}: {r.json()['uuid']}")
```

</details>

### Exercice 2 — Scraper avec rate limiting *(moyen)*

Faire 20 requêtes vers `https://httpbin.org/delay/0.5` en limitant à 5 simultanées avec un sémaphore. Afficher le temps total (attendu ~2s).

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Httpx_async", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import httpx
import time

sem = asyncio.Semaphore(5)

async def fetch(client: httpx.AsyncClient, i: int) -> int:
    async with sem:
        r = await client.get("https://httpbin.org/delay/0.5")
        return r.status_code

start = time.perf_counter()
async with httpx.AsyncClient() as client:
    results = await asyncio.gather(*(fetch(client, i) for i in range(20)))
elapsed = time.perf_counter() - start

print(f"20 requêtes, sem(5) : {elapsed:.2f}s (attendu ≈2s)")
print(f"Tous OK : {all(s == 200 for s in results)}")
```

</details>

### Exercice 3 — Client HTTP avec retry et circuit breaker *(difficile)*

Écrire une classe `ResilientClient` qui :

1. Fait des retries avec backoff exponentiel (max 3).
2. Implémente un **circuit breaker** : après 5 échecs consécutifs, refuse les requêtes pendant 30s.
3. Expose `await client.fetch(url)` comme interface simple.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="03_Httpx_async", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import asyncio
import httpx
import time

class CircuitBreakerOpen(Exception):
    pass

class ResilientClient:
    def __init__(
        self, max_retries: int = 3, failure_threshold: int = 5, reset_timeout: float = 30.0
    ) -> None:
        self._client = httpx.AsyncClient()
        self._max_retries = max_retries
        self._failures = 0
        self._threshold = failure_threshold
        self._reset_timeout = reset_timeout
        self._open_since: float | None = None

    async def fetch(self, url: str) -> httpx.Response:
        # Circuit breaker check
        if self._open_since is not None:
            if time.monotonic() - self._open_since < self._reset_timeout:
                raise CircuitBreakerOpen("Circuit ouvert")
            self._open_since = None
            self._failures = 0

        last_exc = None
        for attempt in range(1, self._max_retries + 1):
            try:
                r = await self._client.get(url, timeout=5.0)
                r.raise_for_status()
                self._failures = 0
                return r
            except (httpx.HTTPError, httpx.TimeoutException) as e:
                last_exc = e
                self._failures += 1
                if self._failures >= self._threshold:
                    self._open_since = time.monotonic()
                    raise CircuitBreakerOpen("Trop d'échecs") from e
                await asyncio.sleep(0.5 * (2 ** (attempt - 1)))
        raise last_exc

    async def close(self) -> None:
        await self._client.aclose()

# Test
client = ResilientClient(max_retries=2, failure_threshold=3)
try:
    r = await client.fetch("https://httpbin.org/get")
    print(f"Succès : {r.status_code}")
finally:
    await client.close()
```

</details>

---

## Ressources

- [docs httpx](https://www.python-httpx.org/)
- [httpx — Async Support](https://www.python-httpx.org/async/)
- [RealPython — Making HTTP Requests with httpx](https://realpython.com/python-httpx/)
- [httpbin.org](https://httpbin.org/) — service de test HTTP